# 04 — The estimator SVC and its parameters

*Notebook 4 of 5 · Support Vector Machines*

You have built every piece by hand: the margin (NB 1), the cost `C` that softens it (NB 2), and the
kernel with its reach `gamma` (NB 3). Now we drive the real `sklearn.svm.SVC` and learn to *set* those
knobs. The centrepiece is the **`C × gamma` bias–variance map** — the one picture that ties the chapter
together — and the practical skills around it: reading the map, handling more than two classes, and
turning an SVM *score* into a calibrated *probability*.

**Prerequisites**
- NB 1–3 (the margin, the cost `C`, the kernel and its reach `gamma`).
- Module 00: NB 09 (over/under-fitting and the U-curve), NB 10 (cross-validation), NB 11 (the
  `Pipeline`); Chapters 02–03 (calibration with `CalibratedClassifierCV`).

**What you'll be able to do**
- Confirm the library `SVC` solves the by-hand problem (parity), then turn its knobs.
- Read the **`C × gamma`** bias–variance map and pick a setting by cross-validation.
- See `gamma` swing the boundary from underfit to overfit.
- Handle **multiclass** problems (one-vs-one), and turn an SVM **score** into a calibrated probability.
- Know when to reach for `LinearSVC`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_course import colors, datasets, viz

viz.use_course_style()
SEED = 0
np.random.seed(SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# the over/underfitting playground (module 00 NB 09 / ch 04 NB 3) and NB 1's separable blobs
Xm, ym = make_moons(n_samples=300, noise=0.30, random_state=SEED)
Xb, yb = make_blobs(
    n_samples=40, centers=[[-2.2, -2.2], [2.2, 2.2]], cluster_std=0.7, random_state=SEED
)
Xb = StandardScaler().fit_transform(Xb)

parity = SVC(kernel="linear", C=1e6).fit(Xb, yb)
parity_svs = sorted(parity.support_.tolist())
print("parity check (SVC linear, C=1e6, on NB 1's separable blobs):")
print(f"  ||w|| = {np.linalg.norm(parity.coef_[0]):.4f}   support vectors at {parity_svs}")
print("  -> identical to NB 1's by-hand street; the library solves the same problem")

## Parity first, then the knobs

The library is not magic: a linear `SVC` with a huge `C` reproduces the exact maximum-margin street we
found by hand in NB 1 — same `‖w‖`, same support vectors. With that trust established, the rest of the
notebook is about the **knobs**: how to set `C` and `gamma`, handle several classes, and read a
probability off the model.

## C and gamma together: the bias–variance map

`C` (NB 2) trades margin width against violations; `gamma` (NB 3) is the RBF's reach. They do not act
alone — together they set the model's effective complexity. The clearest way to see this is a **map**:
cross-validated accuracy over a grid of `C` and `gamma` values.

In [ ]:
Cs = [0.1, 1, 10, 100]
gammas = [0.01, 0.1, 1, 10]
grid = np.array([
    [cross_val_score(make_pipeline(StandardScaler(), SVC(C=C, gamma=g)), Xm, ym, cv=CV).mean()
     for g in gammas]
    for C in Cs
])

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(grid, cmap=colors.CMAP_PROBA, aspect="auto", origin="lower")
ax.set_xticks(range(len(gammas)), [str(g) for g in gammas])
ax.set_yticks(range(len(Cs)), [str(c) for c in Cs])
ax.set_xlabel("gamma (RBF reach)")
ax.set_ylabel("C (cost of violations)")
mid = (grid.max() + grid.min()) / 2
for i in range(len(Cs)):
    for j in range(len(gammas)):
        shade = colors.COLORS["background"] if grid[i, j] > mid else colors.COLORS["text"]
        ax.text(j, i, f"{grid[i, j]:.3f}", ha="center", va="center", color=shade)
fig.colorbar(im, ax=ax, label="cross-validated accuracy")
ax.set_title("The C x gamma bias-variance map (RBF, make_moons)")
ax.grid(False)
plt.show()

**Read the figure.** The left column (small `gamma`) is a flat underfit wall — about 0.83
whatever `C` does, because the RBF's reach is too broad to follow the curve. The bright band is `C`
1–100 with `gamma` ≈ 1: the sweet spot. The top-right corner (large `gamma` *and* large `C`) dims again
(0.937 → 0.910) — the model starts to overfit. This map is the practitioner's view of an SVM; you pick
a cell by **cross-validation**, never by peeking at the test set.

In [ ]:
Xms = StandardScaler().fit_transform(Xm)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, g in zip(axes, [0.01, 1, 10], strict=True):
    clf = SVC(C=1.0, gamma=g).fit(Xms, ym)
    viz.plot_svm_decision(clf, Xms, ym, ax=ax)
    label = {0.01: "underfit", 1: "good fit", 10: "overfit"}[g]
    ax.set_title(f"gamma = {g}  ({label}; {clf.n_support_.sum()} support vectors)")
    ax.set_xlabel("feature 1 (standardized)")
    ax.set_ylabel("feature 2 (standardized)")
fig.tight_layout()
plt.show()

**Read the figure.** At `gamma = 0.01` the reach is so broad the boundary is nearly straight and
underfits — almost every point is a support vector (167). At `gamma = 1` the boundary is a clean curved
street resting on 88 points. At `gamma = 10` each point's reach shrinks to a tiny bubble: the boundary
breaks into wiggly islands that memorize the training set (163 support vectors again). The default
`gamma='scale'` ($1/(n_\text{features}\cdot\mathrm{Var}(X))$) is a sensible starting point — and because
it depends on the features' spread, it is one more reason to standardize (NB 5).

## Kernel and multiclass

The `kernel` (`linear` / `rbf` / `poly`, NB 3) is itself a knob; here we stay on the RBF. And `SVC`
handles **more than two classes** by **one-vs-one**: it trains a separate classifier for each *pair* of
classes and lets them vote. That is different from Chapter 03's softmax/one-vs-rest and from Chapter
04's trees, which are natively multiclass.

In [ ]:
pf = datasets.load_penguins_full()
sub = pf[["bill_length_mm", "flipper_length_mm", "species"]].dropna()
X3 = StandardScaler().fit_transform(sub[["bill_length_mm", "flipper_length_mm"]].to_numpy())
y3 = sub["species"].to_numpy()

ovo = SVC(kernel="rbf").fit(X3, y3)
n_classes = len(ovo.classes_)
cv3 = cross_val_score(SVC(kernel="rbf"), X3, y3, cv=CV).mean()
df_shape = ovo.decision_function(X3[:5]).shape  # 5 rows -> shape (5, n_classes), unambiguous
print(f"species: {list(ovo.classes_)}")
print(f"one-vs-one trains n(n-1)/2 = {n_classes * (n_classes - 1) // 2} pairwise classifiers")
print(f"decision_function shape (aggregated) = {df_shape};  CV accuracy = {cv3:.4f}")

fig = viz.plot_decision_boundary(ovo, X3, y3)
fig.axes[0].set_title("One-vs-one: three species, three pairwise contests, voted")
fig.axes[0].set_xlabel("bill length (standardized)")
fig.axes[0].set_ylabel("flipper length (standardized)")
plt.show()

**Read the figure.** Three species become three pairwise contests (Adélie–Chinstrap,
Adélie–Gentoo, Chinstrap–Gentoo); each is a clean two-class margin problem, and a vote assembles the
three regions shown. (The aggregated `decision_function` returns one score per *class* — shape `(n_samples, 3)`, the `(5, 3)` printed above — not one per pair.) With only a few classes the `n(n-1)/2` count is cheap; the RBF `SVC` reaches CV
0.956 here.

## A score is not a probability

`SVC` outputs a **signed-distance score** (`decision_function`), not a probability — its sign is the
prediction, its magnitude the confidence. When you need an actual probability (to set a threshold, or
weigh a cost), **wrap the model** in `CalibratedClassifierCV`, which fits **Platt scaling** (a sigmoid)
on held-out scores — the same calibration pattern as Chapter 03 NB 6.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.30, stratify=ym, random_state=SEED)

# an uncalibrated sigmoid of the raw score is NOT a true probability
raw = SVC(kernel="rbf").fit(Xtr, ytr)
naive_proba = 1.0 / (1.0 + np.exp(-raw.decision_function(Xte)))
# Platt scaling via CalibratedClassifierCV: ensemble=False refits the SVC on internal CV folds
# and fits the sigmoid on the held-out scores -- the leak-free replacement for probability=True
calibrated = CalibratedClassifierCV(
    SVC(kernel="rbf"), method="sigmoid", ensemble=False
).fit(Xtr, ytr)
platt_proba = calibrated.predict_proba(Xte)[:, 1]

print(f"uncalibrated sigmoid(score) Brier = {brier_score_loss(yte, naive_proba):.4f}")
print(f"Platt-calibrated Brier           = {brier_score_loss(yte, platt_proba):.4f}")
print("note: SVC(probability=True) is deprecated in 1.9 / removed in 1.11;")
print("      use CalibratedClassifierCV(SVC(), ensemble=False) instead")

fig = viz.plot_calibration_curve(yte, naive_proba, n_bins=6, label="uncalibrated score",
                                 color=colors.COLORS["muted"])
viz.plot_calibration_curve(yte, platt_proba, n_bins=6, label="Platt-calibrated",
                           ax=fig.axes[0], color=colors.COLORS["model"])
fig.axes[0].set_title("From SVM score to calibrated probability")
plt.show()

**Read the figure.** The raw score *ranks* points well but does not lie on the diagonal — read as
a probability it is off (Brier 0.106). Platt scaling pulls the reliability curve toward the diagonal and
lowers the Brier to 0.072. Use `CalibratedClassifierCV` for this, **not** the deprecated
`probability=True`; we put a calibrated probability to work setting a clinical threshold in NB 5.

## Two more knobs, named

- **`LinearSVC`** (and `SGDClassifier(loss="hinge")`): a *linear* SVM that scales to large `n` where the
  kernel `SVC` cannot. On the moons it manages only about 0.82 — a straight line cannot follow the curve
  — so it is not the tool *here*; its payoff is the large-data story in NB 5.
- **`class_weight`**: reweights the classes under imbalance, so the margin does not ignore a rare class
  (used lightly in NB 5).

## Tuning honestly

Choose `C` and `gamma` by **cross-validation on the training split**, then read the **sealed test once**
(module 00 NB 10). `GridSearchCV` does the search inside a `Pipeline` so the scaler is refit on each
fold — no leakage.

In [ ]:
search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("svc", SVC())]),
    {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 0.1, 1, 10]},
    cv=CV,
).fit(Xtr, ytr)

print(f"best parameters: {search.best_params_}")
print(f"cross-validated accuracy on train: {search.best_score_:.3f}")
print(f"sealed test accuracy (touched once): {search.score(Xte, yte):.3f}")

**Read the result.** The grid picked the bright-band cell of the map (`C = 10`, `gamma = 1`),
cross-validating to 0.919 on the training split; the sealed test confirms it at 0.944, and we touched
the test set only once. That discipline — search on train, read test once — is the whole of honest
tuning.

## Your turn

1. **Easy — read the map.** From the `C × gamma` heatmap, name one cell that underfits and one that
   overfits, and say in a sentence why each does.
2. **Medium — predict the boundary.** Without running it, describe the boundary you expect at
   `gamma = 0.01` versus `gamma = 10` (both at `C = 1`), and roughly how many support vectors each uses.
3. **Harder — calibrate.** Take a fitted RBF `SVC`, wrap it in `CalibratedClassifierCV(method="sigmoid")`,
   and report whether the Brier score on a held-out set improves over the uncalibrated score.

## What you built

- You confirmed the library `SVC` reproduces NB 1's by-hand street (**parity**), then turned its knobs.
- You read the **`C × gamma` bias–variance map** and picked a setting by cross-validation, and watched
  **`gamma`** swing the boundary from underfit (broad reach, 167 SVs) to overfit (tiny reach, 163 SVs).
- You handled **multiclass** with one-vs-one (three species, three pairwise contests), and turned an SVM
  **score** into a calibrated **probability** with `CalibratedClassifierCV`.
- You tuned honestly with `GridSearchCV` (CV on train, one sealed test), and learned when to reach for
  **`LinearSVC`**.

**Vocabulary:** the `C × gamma` map · `gamma='scale'` · one-vs-one · `decision_function` · Platt scaling
/ `CalibratedClassifierCV` · `LinearSVC`.

## Going further (optional)

`decision_function_shape` switches the aggregated scores between one-vs-rest and one-vs-one views;
`coef0` and `degree` shape the polynomial kernel; `NuSVC` reparametrizes `C` as a bound `nu` on the
fraction of support vectors. The next notebook puts all of this to work on a real diagnostic dataset,
where **standardization** and the **large-`n` limit** of the kernel SVM both bite.

## References

- Cortes C, Vapnik V (1995). *Support-vector networks.* Machine Learning 20:273-297.
  DOI: [10.1007/BF00994018](https://doi.org/10.1007/BF00994018)
- Platt J (1999). *Probabilistic outputs for support vector machines.* Adv. in Large Margin Classifiers,
  MIT Press — the sigmoid calibration `CalibratedClassifierCV(method="sigmoid")` implements.
- Chang C-C, Lin C-J (2011). *LIBSVM.* ACM TIST 2(3):27.
  DOI: [10.1145/1961189.1961199](https://doi.org/10.1145/1961189.1961199) — the solver and its OvO scheme.
- Hsu C-W, Lin C-J (2002). *A comparison of methods for multiclass support vector machines.* IEEE TNN
  13(2):415-425. DOI: [10.1109/72.991427](https://doi.org/10.1109/72.991427)
- Hastie T, Tibshirani R, Friedman J (2009). *The Elements of Statistical Learning*, §12.
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James G, Witten D, Hastie T, Tibshirani R (2021). *An Introduction to Statistical Learning*, §9.
  DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: 03 — The kernel trick · Next: 05 — A demanding case: breast cancer*